<a href="https://colab.research.google.com/github/sadwi18/24P31A4236_DriveReady_OPC_2026/blob/main/Day_1/day1_shopsphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q openai

import os
from google.colab import userdata
from openai import OpenAI

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"


def chat(messages, temperature=0.7, max_tokens=250):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp

In [3]:
slogan_prompt = [
    {"role": "user", "content": "Write a one-line slogan for ShopSphere online store."}
]

print("--- Temperature 0.2 (Consistent) ---")
for i in range(3):
    r = chat(slogan_prompt, temperature=0.2)
    print(f"Run {i+1}: {r.choices[0].message.content.strip()}")

print("\n--- Temperature 1.2 (Creative / Varied) ---")
for i in range(3):
    r = chat(slogan_prompt, temperature=1.2)
    print(f"Run {i+1}: {r.choices[0].message.content.strip()}")

--- Temperature 0.2 (Consistent) ---
Run 1: ShopSphere: Explore, Discover, Shop—All in One Click.
Run 2: 
Run 3: ShopSphere: Explore, Discover, Shop—your world of style, one click at a time.

--- Temperature 1.2 (Creative / Varied) ---
Run 1: ShopSphere: Click into a world of endless choices.
Run 2: ShopSphere: Where every click feels like home.
Run 3: ShopSphere: Where every click sparks a new world of shopping.


In [4]:
hallucination_q = [
    {"role": "user", "content": "Do you offer free same-day express delivery in Vijayawada?"}
]

print("--- Without Policy Facts (Hallucination Check) ---")
r_no_facts = chat(hallucination_q, temperature=0.3)
print(r_no_facts.choices[0].message.content.strip())

grounded_prompt = [
    {
        "role": "system",
        "content": (
            "You are the customer support assistant for ShopSphere Online Store. "
            "Standard shipping takes 3-5 business days; express delivery slots are not guaranteed. "
            "If a policy or delivery guarantee is not explicitly listed, say you will check with the support team."
        ),
    },
    {"role": "user", "content": "Do you offer free same-day express delivery in Vijayawada?"},
]

print("\n--- With Grounding & Escalation Rule ---")
r_grounded = chat(grounded_prompt, temperature=0.3)
print(r_grounded.choices[0].message.content.strip())

--- Without Policy Facts (Hallucination Check) ---
I’m not sure which service you’re referring to, but I don’t have any information about a free same‑day express delivery option in Vijayawada. If you’re looking for a specific courier or e‑commerce platform, it’s best to check their official website or contact their customer support directly. They’ll be able to confirm whether that service is available and what the terms might be. If you let me know the name of the company or any other details, I can try to help you find the right

--- With Grounding & Escalation Rule ---
I’m sorry, but we don’t have a free same‑day express delivery option listed for Vijayawada. Standard shipping typically takes 3‑5 business days. I’ll double‑check with our support team to see if any special arrangements or promotions are available for your area and get back to you shortly. Thank you for reaching out!


In [5]:
# Separate calls (no shared history)
r1 = chat([{"role": "user", "content": "My name is Ananya."}])
r2 = chat([{"role": "user", "content": "What is my name?"}])
print("Separate calls reply:", r2.choices[0].message.content.strip())

# Shared messages list (with memory)
memory_messages = [
    {"role": "user", "content": "My name is Ananya."},
    {"role": "assistant", "content": "Hi Ananya! How can I help you with ShopSphere today?"},
    {"role": "user", "content": "What is my name?"},
]
r3 = chat(memory_messages)
print("Shared history reply:", r3.choices[0].message.content.strip())

Separate calls reply: I’m not sure what your name is—could you let me know?
Shared history reply: You’re Ananya!


In [6]:
q_en = "What is the return window for wireless earbuds and how much do they cost?"
q_te = "వైర్‌లెస్ ఇయర్‌బడ్స్ రిటర్న్ వ్యవధి ఎంత మరియు వాటి ధర ఎంత?"

res_en = chat([{"role": "user", "content": q_en}], temperature=0.2, max_tokens=150)
res_te = chat([{"role": "user", "content": q_te}], temperature=0.2, max_tokens=150)

print("English prompt_tokens:", res_en.usage.prompt_tokens)
print("Telugu prompt_tokens: ", res_te.usage.prompt_tokens)
print("English reply:", res_en.choices[0].message.content.strip())
print("Telugu reply: ", res_te.choices[0].message.content.strip())

English prompt_tokens: 86
Telugu prompt_tokens:  96
English reply: **Return window (typical)**  
| Retailer / Brand | Standard return period | Notes |
|----------------
Telugu reply:  క్షమించండి, మీరు ఏ బ్రాండ్ లేదా మోడల్ గురించి అడుగుతున్నారో నాకు స్పష్టం కాదు. దయచేసి మీరు ఆసక్తి చూపుతున్న వైర్‌ల


In [7]:
SHOPSPHERE_SYSTEM = """
You are the helpful customer support assistant for ShopSphere Online Store.

Store Facts:
1. Products & prices: Wireless Earbuds (Rs. 1499), Smartwatch (Rs. 2499), Laptop Backpack (Rs. 999).
2. Returns: 7-day return and replacement window for unused items in original packaging.
3. Shipping: Standard shipping takes 3-5 business days across India; free shipping on orders Rs. 1500 and above.
4. Payments: UPI, debit/credit cards, and Cash on Delivery are accepted.
5. Support hours: Monday to Saturday, 9:00 AM to 6:00 PM IST.

Rules:
1. Answer only using the store facts above; do not invent prices, discounts, or delivery promises.
2. Keep responses clear, polite, and concise (under 60 words).
3. If a question asks for details outside these facts or order-specific status, state that you will check with the support team.
""".strip()

customer_questions = [
    "How much is a smartwatch and is shipping free on it?",
    "Can I return a laptop backpack after 5 days?",
    "Can you track my order #SH1042 right now?",
]

for idx, q in enumerate(customer_questions, start=1):
    ans = chat(
        [
            {"role": "system", "content": SHOPSPHERE_SYSTEM},
            {"role": "user", "content": q},
        ],
        temperature=0.2,
    )
    print(f"\nQ{idx}: {q}")
    print(f"A{idx}: {ans.choices[0].message.content.strip()}")


Q1: How much is a smartwatch and is shipping free on it?
A1: The Smartwatch costs Rs. 2,499.  
Since the order value exceeds Rs. 1,500, shipping is free.

Q2: Can I return a laptop backpack after 5 days?
A2: Yes, you can return the laptop backpack within 7 days of purchase, as long as it’s unused and in its original packaging.

Q3: Can you track my order #SH1042 right now?
A3: I’m sorry, but I don’t have access to order tracking details. I’ll forward your request to our support team and they’ll get back to you shortly.
